In [ ]:
import shutil
from pathlib import Path

import pandas as pd

from byte_util.util import single_to_double_float, all_sites
from byte_util.met_transport import update_physical_params

rerun = True

# Read in soil and root parameters
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'

soil_params = pd.read_parquet(f'{s3_input_path}/soil_physical_parameters.parquet')
root_params = pd.read_parquet(f'{s3_input_path}/root_water_parameters.parquet')

In [ ]:
# Calculate initial surface flux for each site and simulation type
sim_types = ['spinup', 'hourly', 'daily', 'monthly', 'longterm']

columns = ['site', 'flux'] + sim_types
init_forcing = pd.DataFrame(columns=columns, dtype=float)
init_forcing.set_index(['site', 'flux'], inplace=True)

for site in all_sites:
    forcing_file = f'{s3_base_path}/input-data/processed-data/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Convert from mm/hr to m/s
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    # Convert transpiration from mm/hr to m/d
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24
    # Convert from m/d to 1/d
    dz = 0.01
    met_forcing['transpiration_factor'] = met_forcing['transpiration_m.d'] / dz

    daily = met_forcing.resample('D').mean()
    monthly = met_forcing.resample('MS').mean()

    for col in ['surface_flux_m.s', 'transpiration_factor']:
        # Initial forcing value for spin-up is the long-term mean
        init_forcing.loc[(site, col), 'spinup'] = met_forcing[col].mean()

        # Same for long-term mean
        init_forcing.loc[(site, col), 'longterm'] = met_forcing[col].mean()

        # For hourly, daily, monthly, use the first value in the respective
        # time series. Subsequent values are changed by *.bcvs files
        init_forcing.loc[(site, col), 'hourly'] = met_forcing[col].values[0]
        init_forcing.loc[(site, col), 'daily'] = daily[col].values[0]
        init_forcing.loc[(site, col), 'monthly'] = monthly[col].values[0]

In [ ]:
## Create main MIN3P input file (*.dat)
from min3p.input import InputFile

basepath = Path('../simulations/met_forcing_transport/min3p_runs/base')

for site in all_sites:
    if not rerun:
        continue

    # Create run directories
    sitepath = Path(f'../simulations/met_forcing_transport/min3p_runs/{site}')
    for sim_type in sim_types:
        simpath = sitepath / sim_type
        shutil.rmtree(simpath, ignore_errors=True)
        simpath.mkdir(parents=True)

        infile = InputFile.load('base.dat', path=basepath)

        # First, adjust physical parameters
        update_physical_params(infile, site, soil_params, root_params)

        ## Next, adjust scenario-specific parameters
        # Adjust top boundary to initial rate
        bcvs = infile.boundary_conditions_vsflow
        top_boundary = bcvs.zones[0]
        infil_rate = init_forcing.loc[(site, 'surface_flux_m.s'), sim_type]

        top_boundary.boundary_value = single_to_double_float(f'{infil_rate:0.3e}')

        # Adjust initial transpiration_factor
        for zone in infile.physical_parameters_vsflow.zones:
            rwu = zone.root_water_uptake
            trans_factor = init_forcing.loc[(site, 'transpiration_factor'), sim_type]
            rwu.transpiration_factor = f'{trans_factor:0.6f}'

        if sim_type == 'spinup':
            # Adjust simulation end time
            infile.time_step_control.final_time = 3650.
        else:
            # Adjust simulation end time
            infile.time_step_control.final_time = 3650.

            # Adjust output times for spatial (contour) data
            oc = infile.output_control
            output_times = [float(t) for t in range(365, 3651, 365)]
            oc.output_of_spatial_data = output_times

            # Adjust initial conditions
            infile.initial_conditions_vsflow.text = ("! Initial cond generated from spin-up\n"
                                                     "'read initial condition from file'\n")

            # Adjust initial conditions to include tracers
            icrt = infile.initial_conditions_reactive_transport
            background = icrt.zones[0]
            background.extent_of_zone = [0.0, 1.0, 0.0, 1.0, 0.0, 3.0]
            deep_tracer = ("\n'concentration input'\n\n"
                           "1.0000d-3      'free'         ;'tracer - psi01'\n"
                           "1.0000d-1      'free'         ;'tracer - psi02'\n"
                           "1.0000d-8      'free'         ;'tracer - psi03'\n\n"
                           "'mineral input'\n"
                           "0.00      .true.   'constant' ;phi_init, minequil, update_type -- 'psi03_src'\n"
                           "0.00      1.0d-9   0.00       ;phi_min, log rate constant, unused\n\n"
                           "'extent of zone'\n"
                           "0.0 1.0  0.0 1.0  3.0 3.7\n"
            )
            _ = icrt.add_zone(name='deep tracer', body=deep_tracer)

            shallow_tracer = ("\n'concentration input'\n\n"
                              "1.0000d-1      'free'         ;'tracer - psi01'\n"
                              "1.0000d-3      'free'         ;'tracer - psi02'\n"
                              "1.0000d-8      'free'         ;'tracer - psi03'\n\n"
                              "'mineral input'\n"
                              "0.01      .true.   'constant' ;phi_init, minequil, update_type -- 'psi03_src'\n"
                              "0.00      1.0d-9   0.00       ;phi_min, log rate constant, unused\n\n"
                              "'extent of zone'\n"
                              "0.0 1.0  0.0 1.0  3.7 4.0\n"
            )
            _ = icrt.add_zone(name='shallow tracer', body=shallow_tracer)

            # Adjust concentration of top BC to 1e-3
            bcrt = infile.boundary_conditions_reactive_transport  # Get BC block
            top_conc = bcrt.zones[0].concentration_input  # Get concentrations of top BC
            top_conc.records[0].replace_content("1.0000d-3      'free'")  # Adjust 1st comp (tracer)

            # If sim != longterm or spinup, add transient keywords
            if sim_type != 'longterm':
                bcvs.text += "\n'transient boundary conditions'\n\n"
                infile.physical_parameters_vsflow.transient_transpiration = True

        # Save to file
        sim_file = simpath / f'{sim_type}.dat'
        infile.save(sim_file)

In [ ]:
## Copy over other necessary files (*.rld, *.soi, *.bcvs)
from byte_util.met_transport import create_bcvs_soi, write_transient
from byte_util.util import start_date

for site in all_sites:
    if not rerun:
        continue

    # Create run directories
    sitepath = Path(f'../simulations/met_forcing_transport/min3p_runs/{site}')
    for sim_type in sim_types:
        simpath = sitepath / sim_type

        # Copy over *.rld files
        rldpath = simpath / f'{sim_type}.rld'
        shutil.copy(basepath / 'singleperm.rld', rldpath)

        # Copy over transient files (*.bcvs and *.soi)
        if sim_type in ['hourly', 'daily', 'monthly']:
            forcing_file = f'{s3_base_path}/input-data/processed-data/climate/{site}_hourly_forcing.csv'
            met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

            # Setting delete_first_record to false and dt to 0.001 means that the
            # first timestamp in the bcvs and soi files will be 0.001 d. As a result,
            # the BC and transpiration values entered into the main input file will
            # quickly be overwritten
            bcvs, soi = create_bcvs_soi(met_forcing, freq=sim_type, start_date=start_date,
                                        delete_first_record=False, dt=1e-3)

            # Limit to first 10 years
            mask = soi['time'] < 3651
            soi = soi.loc[mask, :]
            bcvs = bcvs.loc[mask, :]

            # Export to file
            write_transient(simpath / f'{sim_type}.bcvs', bcvs)
            write_transient(simpath / f'{sim_type}.soi', soi)